# Uber Yolculuk Analizi

Bu projede Eylül 2014 Uber NYC verisine bakacağım. Yoğun saatleri çıkarmak istiyorum.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/uber-raw-data-sep14.csv')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
df['Date/Time']=pd.to_datetime(df['Date/Time'])
df['saat']=df['Date/Time'].dt.hour
df['gun']=df['Date/Time'].dt.day_name()
df['saat'].value_counts().sort_index().plot(kind='bar')
plt.title('saate gore')
plt.show()


In [ ]:
sns.heatmap(pd.crosstab(df['Date/Time'].dt.dayofweek,df['saat']),cmap='YlOrRd')
plt.show()


### Boş veri


In [ ]:
df=df.dropna()


### Feature Engineering


In [ ]:
tab=df.groupby(df['Date/Time'].dt.floor('h')).size().reset_index(name='trips')
tab['saat']=tab['Date/Time'].dt.hour
tab['dow']=tab['Date/Time'].dt.dayofweek
x=tab[['saat','dow']]
y=tab['trips']


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


### 3 Model


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

for ad,m in [('LR',LinearRegression()),('DT',DecisionTreeRegressor(random_state=42)),('RF',RandomForestRegressor(n_estimators=80,random_state=42))]:
    m.fit(x_train,y_train)
    print(ad,round(r2_score(y_test,m.predict(x_test)),3))


### Feature Importance + Residual


In [ ]:
rf=RandomForestRegressor(n_estimators=80,random_state=42).fit(x_train,y_train)
print(pd.Series(rf.feature_importances_,index=x.columns))
pred=rf.predict(x_test)
plt.scatter(pred,y_test-pred)
plt.axhline(0,color='r')
plt.show()


### Sonuç

Akşam saatleri yoğun. Saat + gün ile trip sayısı tahmin edilebiliyor. Hedefi tutturdum.
